In [2]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
PREDSDIR   = CONFIGS['filepaths']['predictions']
FIELDVARS  = CONFIGS['experiments']['nn']['runs']['nn_gauss']['fieldvars']
SEEDS      = CONFIGS['experiments']['nn']['seeds']
LATRANGE   = CONFIGS['domain']['latrange']
LONRANGE   = CONFIGS['domain']['lonrange']
SPLIT      = 'test'
NBINS      = 20
MINSAMPLES = 50
MINPRECIP  = 0.1
NBOOT      = 200
LHFCOLORS  = {'Low LHF':'#F2C85E','Middle LHF':'#539ED4','High LHF':'#1F3A93'}
LFCLASSES  = {'Ocean':((0.0,0.1),'#539ED4'),'Coast':((0.1,0.9),'gray6'),'Land':((0.9,1.01),'#A0522D')}
REGISTRY   = {row['name']:dict(form=row['form'],constants=json.loads(row['constants']),train_loss=row['train_loss'],valid_loss=row['valid_loss'])
              for _,row in pd.read_csv(os.path.join(MODELSDIR,'sr','optimized_equations.csv')).iterrows()}

In [4]:
def kernel_integrate(fields,weights,dsig):
    return (fields*weights[None,:,:]*dsig[None,None,:]).sum(axis=2)

def calc_dominant_term(rh,thetae,thetaestar):
    return np.maximum(kappa*(rh-rh0),thetae-gamma*thetaestar-thetac)

def calc_sensible_term(lf,shf):
    return lamshf*(lfc-lf)*(shf-shf0)

def calc_latent_term(lhf):
    return lamlhf*(lhf-lhf0)

def to_precip(exponent):
    return np.maximum(np.expm1(exponent),0.0)

def bin_1d(x,z,edges,minsamples=MINSAMPLES):
    xi     = np.clip(np.digitize(x,edges)-1,0,len(edges)-2)
    counts = np.bincount(xi,minlength=len(edges)-1)
    sums   = np.bincount(xi,weights=z,minlength=len(edges)-1)
    return 0.5*(edges[:-1]+edges[1:]),np.where(counts>=minsamples,sums/np.maximum(counts,1),np.nan)

def calc_slope(x,y,minsamples=MINSAMPLES):
    if x.size<minsamples:
        return np.nan
    xanom = x-x.mean()
    return np.dot(xanom,y-y.mean())/np.dot(xanom,xanom)

def bin_2d(x,y,z,edges,minsamples=MINSAMPLES):
    xi     = np.clip(np.digitize(x,edges[0])-1,0,len(edges[0])-2)
    yi     = np.clip(np.digitize(y,edges[1])-1,0,len(edges[1])-2)
    shape  = (len(edges[0])-1,len(edges[1])-1)
    counts = np.zeros(shape)
    sums   = np.zeros(shape)
    np.add.at(counts,(xi,yi),1)
    np.add.at(sums,(xi,yi),z)
    return np.where(counts>=minsamples,sums/np.maximum(counts,1),np.nan)

def calc_time_sums(X,y,timeidx,ntime):
    p   = X.shape[1]
    xtx = np.stack([np.bincount(timeidx,weights=X[:,i]*X[:,j],minlength=ntime) for i in range(p) for j in range(p)],axis=1).reshape(ntime,p,p)
    xty = np.stack([np.bincount(timeidx,weights=X[:,i]*y,minlength=ntime) for i in range(p)],axis=1)
    return xtx,xty,np.bincount(timeidx,weights=y*y,minlength=ntime),np.bincount(timeidx,weights=y,minlength=ntime),np.bincount(timeidx,minlength=ntime)

def solve_ols(sums,weights):
    xtx,xty,yty,ysum,count = sums
    a    = np.tensordot(weights,xtx,axes=1)
    b    = weights@xty
    beta = np.linalg.solve(a,b)
    sse  = weights@yty-2*beta@b+beta@a@beta
    sst  = weights@yty-(weights@ysum)**2/(weights@count)
    return beta,1-sse/sst

def fit_ols_bootstrap(X,y,timeidx,ntime,nboot=NBOOT,seed=0):
    sums      = calc_time_sums(X,y,timeidx,ntime)
    beta,r2   = solve_ols(sums,np.ones(ntime))
    rng       = np.random.default_rng(seed)
    boots     = np.array([solve_ols(sums,np.bincount(rng.integers(0,ntime,ntime),minlength=ntime).astype(float))[0] for _ in range(nboot)])
    lo,hi     = np.percentile(boots,[2.5,97.5],axis=0)
    return beta,lo,hi,r2

In [5]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
c3,c4,c5 = (REGISTRY['sr_atm_eq']['constants'][name] for name in ['c3','c4','c5'])
c6,c7,c8 = (REGISTRY['sr_sfc_eq']['constants'][name] for name in ['c6','c7','c8'])

eta    = STATS['tp_std']*c3/STATS['thetae_std']**3
gamma  = c4*STATS['thetae_std']/STATS['thetaestar_std']
thetac = STATS['thetae_mean']-gamma*STATS['thetaestar_mean']+c5*STATS['thetae_std']
kappa  = STATS['thetae_std']/STATS['rh_std']
rh0    = STATS['rh_mean']

lamshf = STATS['tp_std']*c6/STATS['shf_std']
lfc    = c7
shf0   = STATS['shf_mean']
lamlhf = STATS['tp_std']*c8/STATS['lhf_std']
lhf0   = STATS['lhf_mean']

constdf = pd.DataFrame([
    {'Constant':'$\\lambda_\mathrm{S}$','Value':f'{lamshf:.3e}','Units':'m²/W'},
    {'Constant':'$\\mathrm{LF}_c$','Value':f'{lfc:.2f}','Units':'dimensionless'},
    {'Constant':'$\\mathrm{SHF}_0$','Value':f'{shf0:.2f}','Units':'W/m²'},
    {'Constant':'$\\lambda_\mathrm{L}$','Value':f'{lamlhf:.3e}','Units':'m²/W'},
    {'Constant':'$\\mathrm{LHF}_0$','Value':f'{lhf0:.2f}','Units':'W/m²'}])
display(constdf.style.hide(axis='index'))

Constant,Value,Units
$\lambda_\mathrm{S}$,1.250e-02,m²/W
$\mathrm{LF}_c$,0.74,dimensionless
$\mathrm{SHF}_0$,13.91,W/m²
$\lambda_\mathrm{L}$,1.564e-03,m²/W
$\mathrm{LHF}_0$,124.39,W/m²


In [6]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime,nlat,nlon = (ds.sizes[dim] for dim in ['time','lat','lon'])
    lat,lon,dsig    = ds['lat'].values,ds['lon'].values,ds['dsig'].values
    fields = np.stack([ds[name].transpose('time','lat','lon','sig').values.reshape(-1,dsig.size) for name in FIELDVARS],axis=1)
    tp,shf,lhf = (ds[name].transpose('time','lat','lon').values.ravel() for name in ['tp','shf','lhf'])
    lf     = xr.broadcast(ds['lf'],ds['tp'])[0].transpose('time','lat','lon').values.ravel()
with xr.open_dataset(os.path.join(PREDSDIR,f'sr_sfc_eq_{SPLIT}_predictions.nc')) as ds:
    savedtp = ds['tp'].squeeze().transpose('time','lat','lon').values.ravel()

kernels = []
for seed in SEEDS:
    with xr.open_dataset(os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc'),engine='h5netcdf') as ds:
        kernels.append(ds['k'].values)
integrals    = dict(zip(FIELDVARS,kernel_integrate(fields,np.mean(kernels,axis=0),dsig).T))
dominant     = calc_dominant_term(integrals['rh'],integrals['thetae'],integrals['thetaestar'])
sensibleterm = calc_sensible_term(lf,shf)
latentterm   = calc_latent_term(lhf)
surfaceterm  = sensibleterm+latentterm
atmtp        = to_precip(eta*dominant**3)
sfctp        = to_precip(eta*dominant**3+surfaceterm)
finite       = np.isfinite(dominant)&np.isfinite(surfaceterm)&np.isfinite(tp)
favorable    = finite&(atmtp>=MINPRECIP)
missed       = tp-atmtp
added        = sfctp-atmtp
print(f'Max difference from saved SR-SFC predictions: {np.nanmax(np.abs(sfctp[finite]-savedtp[finite])):.2e} mm')

lfmasks = {name:(lf>=bounds[0])&(lf<bounds[1]) for name,(bounds,_) in LFCLASSES.items()}
colors  = {name:color for name,(_,color) in LFCLASSES.items()}

Max difference from saved SR-SFC predictions: 8.94e-02 mm


In [ ]:
regions     = {'Ocean':(favorable&(lf<0.5),'#539ED4',0.0),'Land':(favorable&(lf>=0.5),'#D42028',1.0)}
regionmasks = {region:mask for region,(mask,_,_) in regions.items()}
sedges      = np.linspace(*np.percentile(surfaceterm[favorable],[1,99]),NBINS+1)
fluxbins    = {}
for region,(mask,_,lfvalue) in regions.items():
    edges = (np.linspace(*np.percentile(shf[mask],[1,99]),NBINS+1),np.linspace(*np.percentile(lhf[mask],[1,99]),NBINS+1))
    zeroshf = np.linspace(edges[0][0],edges[0][-1],100)
    fluxbins[region] = dict(centers=[0.5*(edge[:-1]+edge[1:]) for edge in edges],
                            limits=[(edge[0],edge[-1]) for edge in edges],
                            zeroline=(zeroshf,lhf0-lamshf*(lfc-lfvalue)/lamlhf*(zeroshf-shf0)),
                            needed=bin_2d(shf[mask],lhf[mask],missed[mask],edges),
                            made=bin_2d(shf[mask],lhf[mask],added[mask],edges))
corrlim = np.nanpercentile(np.abs([data[key] for data in fluxbins.values() for key in ['needed','made']]),98)

fig,axs = pplt.subplots(array=[[1,1],[2,3],[4,5]],figwidth=5,share=False,span=False)
for region,(mask,color,_) in regions.items():
    centers,era5mean = bin_1d(surfaceterm[mask],tp[mask],sedges)
    _,sfcmean        = bin_1d(surfaceterm[mask],sfctp[mask],sedges)
    _,atmmean        = bin_1d(surfaceterm[mask],atmtp[mask],sedges)
    axs[0].scatter(centers,era5mean,color=color,markersize=20,zorder=3)
    axs[0].plot(centers,sfcmean,color=color,linewidth=1,label=region)
    axs[0].plot(centers,atmmean,color=color,linestyle='--',linewidth=1)
for label,kwargs in [('ERA5',dict(marker='o',linestyle='none',markersize=3)),('SR-SFC',dict(linewidth=1)),('SR-ATM',dict(linestyle='--',linewidth=1))]:
    axs[0].plot([],[],color='black',label=label,**kwargs)
axs[0].axvline(0,color='gray',linewidth=0.5)
axs[0].format(title='Precipitation Response to Surface Conditions',xlabel='$\\mathit{S}$',ylabel='Total Precipitation (mm)')
axs[0].legend(loc='ul',ncols=1)

for row,(region,data) in enumerate(fluxbins.items()):
    for col,key in enumerate(['needed','made']):
        ax = axs[1+2*row+col]
        mcorr = ax.pcolormesh(*data['centers'],data[key].T,cmap='ColdHot_r',vmin=-corrlim,vmax=corrlim,levels=16,extend='both')
        ax.plot(*data['zeroline'],color='k',linewidth=0.8,linestyle='--')
        ax.format(xlim=data['limits'][0],ylim=data['limits'][1],xlabel='SHF (W/m$^2$)',facecolor='gray1')
    axs[1+2*row].format(ylabel=f'{region}\nLHF (W/m$^2$)')
    axs[2+2*row].format(yticklabels=[])
axs[1].format(title='Correction Needed (ERA5 $-$ SR-ATM)')
axs[2].format(title='Correction Made (SR-SFC $-$ SR-ATM)')
fig.format(abc=True,titleloc='l',grid=False)
fig.colorbar(mcorr,loc='b',cols=(1,2),label='Precipitation Difference (mm)')
pplt.show()
fig.save('../figs/fig_5.jpg')

In [11]:
timeidx = np.repeat(np.arange(ntime),nlat*nlon)
fmt     = lambda value,lo,hi:f'{value:.3f} [{lo:.3f}, {hi:.3f}]'
fitrows,sloperows = [],[]
for region,mask in regionmasks.items():
    logresid = np.log1p(tp[mask])-np.log1p(atmtp[mask])
    zshf     = (shf[mask]-shf[mask].mean())/shf[mask].std()
    zlhf     = (lhf[mask]-lhf[mask].mean())/lhf[mask].std()
    ones     = np.ones_like(zshf)
    for model,X in [('SHF + LHF',np.column_stack([ones,zshf,zlhf])),('SHF + LHF + SHF×LHF',np.column_stack([ones,zshf,zlhf,zshf*zlhf]))]:
        beta,lo,hi,r2 = fit_ols_bootstrap(X,logresid,timeidx[mask],ntime)
        fitrows.append({'Region':region,'Model':model,'SHF':fmt(beta[1],lo[1],hi[1]),'LHF':fmt(beta[2],lo[2],hi[2]),
                        'SHF×LHF':fmt(beta[3],lo[3],hi[3]) if X.shape[1]==4 else '','R²':f'{r2:.4f}'})
    bounds = np.percentile(lhf[mask],[0,100/3,200/3,100])
    for label,lo,hi in zip(LHFCOLORS,bounds[:-1],bounds[1:]):
        sub = (lhf[mask]>=lo)&(lhf[mask]<=hi)
        beta,blo,bhi,_ = fit_ols_bootstrap(np.column_stack([ones[sub],zshf[sub]]),logresid[sub],timeidx[mask][sub],ntime)
        sloperows.append({'Region':region,'LHF Tercile':f'{label} ({lo:.0f} to {hi:.0f} W/m²)','SHF Slope':fmt(beta[1],blo[1],bhi[1])})
display(pd.DataFrame(fitrows).style.hide(axis='index'))
display(pd.DataFrame(sloperows).style.hide(axis='index'))

Region,Model,SHF,LHF,SHF×LHF,R²
Ocean,SHF + LHF,"0.170 [0.167, 0.173]","0.036 [0.033, 0.039]",,0.2107
Ocean,SHF + LHF + SHF×LHF,"0.166 [0.162, 0.169]","0.036 [0.033, 0.039]","0.010 [0.008, 0.012]",0.2115
Land,SHF + LHF,"-0.106 [-0.112, -0.099]","0.142 [0.134, 0.149]",,0.0387
Land,SHF + LHF + SHF×LHF,"-0.077 [-0.083, -0.071]","0.169 [0.160, 0.176]","-0.056 [-0.061, -0.051]",0.0473


Region,LHF Tercile,SHF Slope
Ocean,Low LHF (0 to 119 W/m²),"0.155 [0.148, 0.160]"
Ocean,Middle LHF (119 to 158 W/m²),"0.170 [0.166, 0.174]"
Ocean,High LHF (158 to 837 W/m²),"0.183 [0.179, 0.187]"
Land,Low LHF (-28 to 26 W/m²),"0.084 [0.067, 0.102]"
Land,Middle LHF (26 to 96 W/m²),"-0.066 [-0.072, -0.061]"
Land,High LHF (96 to 508 W/m²),"-0.088 [-0.094, -0.080]"
